# Hyperliquid Perpetual Futures Data Import

This notebook imports:
1. **OHLC data** (prices and volume) for Hyperliquid perpetual futures
2. **Funding rate data** (hourly funding rates)

## About Hyperliquid
- **Type**: Decentralized perpetual futures exchange
- **Available**: Globally (no geo-restrictions like CEX with BaFin)
- **Contracts**: 229+ perpetual futures
- **Settlement**: USDC (all contracts are linear/USDC-margined)
- **Funding**: Hourly funding rates

## API Details
- **Endpoint**: https://api.hyperliquid.xyz/info
- **Ticker Format**: Simple coin names (BTC, ETH, SOL)
- **Data Range**: Historical data from 2023+
- **Rate Limit**: ~200 req/min

In [ ]:
from clients.hyperliquid_client import HyperliquidClient
from clients.db_client import DBClient
import pandas as pd
from datetime import datetime

In [ ]:
# Configuration
EXCHANGE = 'hyperliquid'
INTERVAL = '1h'  # 1 hour candles
START_DATE = '2025-01-01' # Only the most recent 5000 candles are available for the OHLCV
END_DATE = datetime.today().strftime('%Y-%m-%d') # Use today's date as the end date

hl = HyperliquidClient()

## Test Download - Single Contract

In [ ]:
# Test with Bitcoin perpetual
test_symbol = 'BTC'

# Test OHLC download
df_ohlc = hl.download_ohlc(
    symbol=test_symbol,
    interval=INTERVAL,
    start_date=START_DATE,
    end_date=END_DATE
)

print(f"\nOHLC Data Preview:")
print(df_ohlc.head())
print(f"\nShape: {df_ohlc.shape}")
print(f"Date range: {df_ohlc['timestamp'].min()} to {df_ohlc['timestamp'].max()}")

In [ ]:
# Test funding rate download
df_funding = hl.download_funding_rates(
    symbol=test_symbol,
    start_date=START_DATE,
    end_date=END_DATE
)

In [ ]:
print(f"Funding Rate Data Preview:")
print(df_funding.head())
print(f"Date range: {df_funding['timestamp'].min()} to {df_funding['timestamp'].max()}")

## Import OHLC data for all Hyperliquid perpetual futures

In [ ]:
with DBClient() as db:
    instruments_dict = db.get_perpetuals_dict(EXCHANGE)


In [ ]:
with DBClient() as db:
    for ticker, instrument_id in instruments_dict.items():
        print(f"\nProcessing {ticker}...")
        
        # Download OHLC data
        df_ohlc = hl.download_ohlc(
            symbol=ticker,
            interval=INTERVAL,
            start_date=START_DATE,
            end_date=END_DATE
        )
        
        if df_ohlc.empty:
            print(f"No OHLC data available for {ticker}")
            continue
        
        # Check existing data range
        min_ts, max_ts = db.get_timestamp_range(instrument_id)
        
        # Check for duplicates
        if min_ts and max_ts:
            # Filter out overlapping data
            df_ohlc = df_ohlc[(df_ohlc['timestamp'] < min_ts) | (df_ohlc['timestamp'] > max_ts)]
                
            if df_ohlc.empty:
                print(f"All data already exists, skipping")
                continue

        # Add instrument_id and insert
        df_ohlc['instrument_id'] = instrument_id
        rows_inserted = db.insert_market_data(df_ohlc)
        
        if rows_inserted > 0:
            print(f"Inserted {rows_inserted:,} OHLC records")
        else:
            print(f"No new OHLC records inserted (duplicates?)")

    print("OHLC IMPORT COMPLETED")

## Import funding rate data for all Hyperliquid perpetual futures

In [ ]:
with DBClient() as db:
    for ticker, instrument_id in instruments_dict.items():
        print(f"\nProcessing funding for {ticker}...")
        
        # Download funding rate data
        df_funding = hl.download_funding_rates(
            symbol=ticker,
            start_date=START_DATE,
            end_date=END_DATE
        )
        
        if df_funding.empty:
            print(f"No funding data available for {ticker}")
            continue
        
        # Check existing funding data range
        min_ts, max_ts = db.get_funding_timestamp_range(instrument_id)
        
        # Check for duplicates
        if min_ts and max_ts:
            # Filter out overlapping data
            df_funding = df_funding[(df_funding['timestamp'] < min_ts) | (df_funding['timestamp'] > max_ts)]
                
            if df_funding.empty:
                print(f"All funding data already exists for {ticker}, skipping")
                continue

        # Add instrument_id and insert
        df_funding['instrument_id'] = instrument_id
        rows_inserted = db.insert_funding_data(df_funding)
        
        if rows_inserted > 0:
            print(f"Inserted {rows_inserted:,} funding records for {ticker}")
        else:
            print(f"No new funding records inserted for {ticker} (duplicates?)")

    print("FUNDING RATE IMPORT COMPLETED")